# Mass Lidar Processing (.laz)

In [1]:
import geopandas as gpd
from pathlib import Path

cloud_dir = Path("datasets/wifire_ca_gaps")
gdf = gpd.read_file(cloud_dir / "one/spatial_extents_ca_californiagaps_1_b23.json")
print(gdf.shape)
gdf.head()

(4486, 8)


,name,package_id,description_id,temporal,extra_links,crs,url,geometry
0,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF658087,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.2236 36.92476, -121.21484 36.92..."
1,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF658088,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.22304 36.92674, -121.2148 36.92..."
2,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF659086,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.20974 36.91554, -121.2085 36.91..."
3,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF659087,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.21505 36.92463, -121.20362 36.9..."
4,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF659088,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.21484 36.9292, -121.20351 36.92..."


In [2]:
x = gdf.iloc[0]
x

name              USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...
package_id                                  CA_CaliforniaGaps_1_B23
description_id                                          10SFF658087
temporal          { "endTime": "20231129", "startTime": "2023102...
extra_links       [ { "onlink": "http:\/\/www.asprs.org\/Committ...
crs                                                       epsg:4326
url               https://rockyweb.usgs.gov/vdelivery/Datasets/S...
geometry          POLYGON ((-121.2235962696 36.9247608381, -121....
Name: 0, dtype: object

In [3]:
x["url"]

'https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/CA_CaliforniaGaps_B23/CA_CaliforniaGaps_1_B23/LAZ/USGS_LPC_CA_CaliforniaGaps_B23_10SFF658087.laz'

In [ ]:
import requests

res = requests.get(x["url"])
res

<Response [200]>

In [20]:
data_dir = Path("datasets") / "mass_lidar" 

def save_laz_file(id, url):
    res = requests.get(url)
    # wb since zip uses binary
    with open(data_dir/ f"lid_{id}.laz", "wb") as f:
        f.write(res.content)


In [21]:
save_laz_file(0, x["url"])

In [12]:
import laspy
import numpy as np

las = laspy.read(data_dir / "lid_1.laz")
las

<LasData(1.4, point fmt: <PointFormat(6, 0 bytes of extra dims)>, 6315207 points, 2 vlrs)>

In [17]:
gdf["ix"] = gdf.index
gdf.head(3)

,name,package_id,description_id,temporal,extra_links,crs,url,geometry,ix
0,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF658087,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.2236 36.92476, -121.21484 36.92...",0
1,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF658088,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.22304 36.92674, -121.2148 36.92...",1
2,USGS Lidar Point Cloud CA_CaliforniaGaps_B23 1...,CA_CaliforniaGaps_1_B23,10SFF659086,"{ ""endTime"": ""20231129"", ""startTime"": ""2023102...","[ { ""onlink"": ""http:\/\/www.asprs.org\/Committ...",epsg:4326,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,"POLYGON ((-121.20974 36.91554, -121.2085 36.91...",2


In [22]:
gdf.apply(lambda r: save_laz_file(r["ix"], r["url"]), axis=1)

KeyboardInterrupt: 

# Processing

In [2]:
import laspy
import numpy as np
from pathlib import Path
import open3d as o3d

data_dir = Path("datasets") / "mass_lidar" 
las = laspy.read(data_dir / "lid_1.laz")
las

<LasData(1.4, point fmt: <PointFormat(6, 0 bytes of extra dims)>, 1514242 points, 2 vlrs)>

# DBSCAN (clustering)

In [3]:
labels = las.classification
print(set(labels))

# 3. Define your RGB mapping (Values must be between 0.0 and 1.0)
color_map = {
    1: [0.0, 0.8, 0.0],  # Trees/High Veg -> Green
    2: [0.6, 0.4, 0.2],  # Ground -> Brown
    7: [0.8, 0.1, 0.1],  # Lowpoint -> Red
}

# 4. Create an empty color array and populate it
colors = np.zeros((len(labels), 3)) # Default to black (0, 0, 0)

for class_id, rgb in color_map.items():
    # Find all points matching this class and apply the color
    mask = (labels == class_id)
    colors[mask] = rgb

# Optional: Set a default color (like gray) for any unmapped points
unmapped_mask = ~np.isin(labels, list(color_map.keys()))
colors[unmapped_mask] = [0.5, 0.5, 0.5]

{np.uint8(1), np.uint8(2), np.uint8(7)}


In [4]:
point_data = np.stack([las.x, las.y, las.z], axis=0).transpose()
geom = o3d.geometry.PointCloud()
geom.colors = o3d.utility.Vector3dVector(colors)
geom.points = o3d.utility.Vector3dVector(point_data)

o3d.visualization.draw_geometries([geom])

In [9]:
non_ground_mask = (np.array(las.classification) != 2)
filtered_points = point_data[non_ground_mask]
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(filtered_points)
# 1m isn't perfect, but it doesn't leave blank spots or cluster too large an object
labels = np.array(pcd.cluster_dbscan(eps=1, min_points=20, print_progress=True))
max_label = labels.max()
print(f"Point cloud has {max_label + 1} distinct objects")

Point cloud has 4729 distinct objects


In [10]:
import matplotlib.pyplot as plt

colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([pcd])

# HDBScan (automatically estimate different optimal cluster sizes)

In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
from sklearn.cluster import HDBSCAN

# Assuming 'filtered_points' is your (N, 3) numpy array containing only trees/bushes

# 1. Initialize HDBSCAN
# min_cluster_size: The smallest number of points you would consider a valid bush.
# min_samples: Controls noise. A higher number makes the algorithm more conservative, 
# dropping more sparse boundary points as noise (-1).
clusterer = HDBSCAN(min_cluster_size=20, min_samples=10)

# 2. Fit the model and extract labels
print("Running HDBSCAN...")
labels = clusterer.fit_predict(filtered_points)

max_label = labels.max()
print(f"HDBSCAN found {max_label + 1} distinct objects")

# 3. Create dynamic colors
# Normalize labels to fit within the colormap (0.0 to 1.0)
# Adding a small fallback if only 1 cluster is found to avoid division by zero
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))

# Force all noise points (label -1) to be black
colors[labels < 0] = [0, 0, 0, 1] 

# 4. Apply to Open3D and visualize
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(filtered_points)
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])

o3d.visualization.draw_geometries([pcd])

Running HDBSCAN...


c:\Users\seani\miniforge3\envs\shrub\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


In [2]:
from cuml.cluster import HDBSCAN

ModuleNotFoundError: No module named 'cuml'